In [1]:
!unzip archive\ \(2\).zip

Archive:  archive (2).zip
  inflating: creditcard.csv          


In [2]:
import pandas as pd

# This reads the file
df = pd.read_csv('creditcard.csv')

# This shows the first 5 rows of your data
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
print("Total Transactions:", len(df))
print("Fraudulent Transactions:", df['Class'].sum())

Total Transactions: 284807
Fraudulent Transactions: 492


In [4]:
from sklearn.preprocessing import StandardScaler

# 1. We remove 'Time' because it's not needed for now
df = df.drop(['Time'], axis=1)

# 2. We make the 'Amount' values smaller/standardized
df['Amount'] = StandardScaler().fit_transform(df['Amount'].values.reshape(-1, 1))

# 3. Show the cleaned data
df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0.244964,0
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,-0.342475,0
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,1.160686,0
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0.140534,0
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,-0.073403,0


In [5]:
from sklearn.model_selection import train_test_split

# X is the data (features), y is the answer (Class)
X = df.drop(['Class'], axis=1)
y = df['Class']

# We give 80% to training and 20% to testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training pile size:", len(X_train))
print("Testing pile size:", len(X_test))

Training pile size: 227845
Testing pile size: 56962


In [10]:
from sklearn.linear_model import LogisticRegression

# 1. Create a faster model
# We use 'max_iter=1000' to give it enough time to find the pattern
fast_model = LogisticRegression(max_iter=1000)

# 2. Start the Training
fast_model.fit(X_train, y_train)

print("Fast training finished!")

Fast training finished!


In [11]:
from sklearn.metrics import confusion_matrix, classification_report

# 1. Ask the AI to predict
y_pred = fast_model.predict(X_test)

# 2. Show the results
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))
print("\n--- Detailed Report ---")
print(classification_report(y_test, y_pred))

--- Confusion Matrix ---
[[56854    10]
 [   42    56]]

--- Detailed Report ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.85      0.57      0.68        98

    accuracy                           1.00     56962
   macro avg       0.92      0.79      0.84     56962
weighted avg       1.00      1.00      1.00     56962



In [12]:
from imblearn.over_sampling import SMOTE

# 1. This creates synthetic fraud cases so the AI has more to study
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 2. Train the fast model again with this new "balanced" data
balanced_model = LogisticRegression(max_iter=1000)
balanced_model.fit(X_train_smote, y_train_smote)

# 3. Test it again
y_pred_new = balanced_model.predict(X_test)

print("--- New Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred_new))
print("\n--- New Detailed Report ---")
print(classification_report(y_test, y_pred_new))

--- New Confusion Matrix ---
[[55411  1453]
 [    8    90]]

--- New Detailed Report ---
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.99     56962

